# Neuron interference: P/N and prediction changes

Teacher 8; students 8 and 16; both layers train. This notebook analyzes existing
SGD states without training. The main comparison uses 40 paired early seeds;
later matched-loss comparisons use eight (lr 1) and five (lr 0.1).

**Finding:** more early positive contributions and more partial cancellation at
width 16, but net prediction movements still reinforce. Lower full-step collateral
harm does not persist clearly at 1% step length. See
[the report](../docs/neuron_interference.md) for definitions and limitations.


In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Image, display

ROOT = Path.cwd()
if not (ROOT / 'pyproject.toml').exists():
    ROOT = ROOT.parent
OUT = ROOT / 'runs/pier_study/neurons'
pn = pd.read_csv(OUT / 'gradient_masses_summary.csv')
pn_effects = pd.read_csv(OUT / 'gradient_masses_effects.csv')
movement = pd.read_csv(OUT / 'prediction_changes_summary.csv')
effects = pd.read_csv(OUT / 'prediction_changes_effects.csv')
selected = pd.read_csv(OUT / 'selected_states.csv')


## 1. Positive and negative demands through each neuron

$q_r=g_{B,r}^\top g_{C,r}$,
$P=\mathbb E\sum_r[q_r]_+/S$ and $N=\mathbb E\sum_r[-q_r]_+/S$,
where $S=\mathbb E\|g_B\|^2$. Signs are taken before summing neurons.
Whole-network normalization avoids making a wider network look better simply
because each neuron contributes less. These are conflicts between data demands
through one neuron, not direct inner products between different neurons.


In [ ]:
primary_pn = pn[(pn.comparison_kind == 'matched_loss') &
                ((pn.comparison != 'mse_0.2') | (pn.probes == 512))]
display(primary_pn[['optimizer','comparison','width','n_seeds','train_mse_mean',
                    'P','N','global_N','offset_negative']].round(6))
display(pn_effects[(pn_effects.comparison == 'mse_0.2') &
                   (pn_effects.probes == 512)][['metric','n','mean','low','high']].round(6))
display(Image(filename=str(OUT / 'neuron_PN.png')))


The early increase in P is clearer than any change in N. Near MSE .01,
negative neuron mass increases with width. Equal-step curves must be read alongside
their different training losses; late ratios may involve numerical floors.

The following seed-0 illustration retains individual neuron contributions.
Neuron numbers identify positions within each model and are not matched across widths.


In [ ]:
units = pd.read_csv(OUT / 'gradient_masses_neurons.csv')
example = units[(units.optimizer == 'sgd_1') & (units.comparison == 'mse_0.2') &
                (units.probes == 512) & (units.seed == 0)]
fig, axes = plt.subplots(1,2,figsize=(10,3.3),sharey=True)
for ax, width in zip(axes,[8,16]):
    part = example[example.width == width].sort_values('neuron')
    ax.bar(part.neuron, part.P, color='#218a65',label='positive contribution')
    ax.bar(part.neuron, -part.N,color='#cf5252',label='negative contribution')
    ax.axhline(0,color='black',lw=.6)
    ax.set(title=f'Width {width}: seed 0, early state',xlabel='neuron index')
axes[0].set_ylabel('contribution / whole-network gradient energy')
axes[0].legend(frameon=False,fontsize=8)
fig.tight_layout()
plt.show()


## 2. Exact prediction changes and protective cancellation

For each candidate update, $d_r(x)$ is the exact change of neuron r's output,
and $d=\sum_r d_r$. With residual $e=f-y$, isolated half-MSE changes are
$h_r=e d_r+\frac12d_r^2$, while the joint change is
$h=e d+\frac12d^2$. The isolated changes do not add to the joint change.

Partial cancellation is $C=1-\mathbb E|d|/\mathbb E\sum_r|d_r|$.
Net movement interaction is $R=\mathbb E d^2/\mathbb E\sum_r d_r^2$.
R below one means net cancellation; R above one means net reinforcement.
R_perpendicular applies the same ratio after removing movement along the current
test residual vector.

Accurate examples satisfy $|e|\leq .05$ at **both widths**, so each seed's
comparison evaluates exactly the same examples. The offset fraction weights
hypothetical one-neuron harmful changes on examples the joint update does not harm.


In [ ]:
matched = movement[(movement.comparison_kind == 'matched_loss') &
                   (movement.scale == 1) & movement.subset.isin(['all','well_fit_0.05'])]
display(matched[['optimizer','comparison','subset','width','n_seeds','n_examples_mean',
                 'R','R_perpendicular','cancellation_fraction','solo_harm_offset_fraction',
                 'joint_harm_per_source_gain','joint_benefit_per_source_gain']].round(5))
display(Image(filename=str(OUT / 'neuron_prediction_interactions.png')))


More partial cancellation coexists with R > 1 at both widths. Therefore
some neuron disturbances offset each other, but cross-neuron terms reinforce
prediction movement on balance. Increasing the number of terms itself provides
more opportunities for partial cancellation; these observations do not establish
learned coordination or explain the whole training advantage.

## Step length and actual collateral loss

H/U is the mean positive held-out loss change divided by mean source-batch
improvement. G/U uses the negative loss changes. The step lengths have not been
chosen to make source improvements exactly identical. The 1% check shortens
directions at saved states; it does not retrain with a lower learning rate.


In [ ]:
early = movement[(movement.optimizer == 'sgd_1') &
                 (movement.comparison == 'mse_0.2') &
                  movement.subset.isin(['all','well_fit_0.05'])]
display(early[['scale','subset','width','joint_harm_per_source_gain',
               'joint_benefit_per_source_gain','cancellation_fraction']].round(6))
display(Image(filename=str(OUT / 'neuron_step_sensitivity.png')))
display(effects[(effects.optimizer == 'sgd_1') & (effects.comparison == 'mse_0.2') &
                (effects.subset == 'well_fit_0.05') &
                 effects.metric.isin(['cancellation_fraction','joint_harm_per_source_gain'])]
        [['scale','metric','n','mean','low','high']].round(6))


## Fresh source batches and independent verification

The fresh check draws 32 further minibatches with seed 20261018 and recomputes
their gradients. Intervals resample training seeds, condition on the fixed
teacher/data, and are exploratory pointwise intervals.


In [ ]:
fresh = pd.read_csv(OUT / 'validation_prediction_changes_summary.csv')
display(fresh[['scale','subset','width','R','cancellation_fraction',
               'solo_harm_offset_fraction','joint_harm_per_source_gain']].round(6))
display(json.loads((OUT / 'audit.json').read_text()))


## Inspect one prediction

This illustration fixes seed 0 and source batch 0, then takes the first test
example meeting the common accuracy threshold. It is not selected for showing
cancellation. Each bar is one neuron's exact prediction change; green/red means
that isolated update helps/harms this example. The dashed line is the total
prediction change, with its actual joint loss change in the title.


In [ ]:
examples = {w: np.load(OUT / f'example_w{w}_seed0.npz') for w in [8,16]}
index = int(np.flatnonzero(examples[8]['well_fit'])[0])
fig, axes = plt.subplots(1,2,figsize=(11,3.6),sharey=True)
for ax, width in zip(axes,[8,16]):
    example = examples[width]
    d = example['moves'][0,index]
    e = example['residual'][index]
    solo = e*d + .5*d*d
    total = d.sum()
    joint = e*total + .5*total*total
    ax.bar(np.arange(width),d,color=np.where(solo>0,'#cf5252','#218a65'))
    ax.axhline(total,ls='--',color='black',lw=1,label='total prediction change')
    ax.axhline(0,color='gray',lw=.6)
    ax.set(title=f'Width {width}: residual {e:.3g}; joint Δloss {joint:.3g}',xlabel='neuron index')
axes[0].set_ylabel('exact prediction change')
axes[0].legend(frameon=False,fontsize=8)
fig.suptitle(f'Seed 0, source batch 0, common accurate test example {index}')
fig.tight_layout()
plt.show()
